#### Travail sur le Catalogue
- A partir des zips, on veut extraire les fichiers
- Suivi sur un fichier csv et vérification dans un premier temps si doublons

In [ ]:
import zipfile
from pathlib import Path

In [ ]:
Path.cwd()

In [ ]:
chemin_zip_raw = Path.cwd().parent / "data" / "raw" / "best_of"
print(chemin_zip_raw)

In [ ]:
chemin_destination = Path.cwd().parent / "data" / "test_1"
print(chemin_destination)

In [ ]:
list(chemin_zip_raw.iterdir())

In [ ]:
fichier_source = chemin_zip_raw / "Bestof_test.zip"
dossier_destination = chemin_destination

with zipfile.ZipFile(fichier_source, "r") as zip_ref:
    zip_ref.extractall(dossier_destination)

    print("fichier zip décompressé avec succès !")

<h1 style = "color: red; text-align: center">RECUPERATION DES DONNEES DES IMAGES POUR CLASSER DANS TABLEUR</h1>

In [ ]:
import hashlib

from PIL import Image
from PIL.ExifTags import TAGS

In [ ]:
def calculer_hash(image_path):
    sha256 = hashlib.sha256()

    with open(image_path, "rb") as f:
        for bloc in iter(lambda: f.read(4096), b""):
            sha256.update(bloc)

    return sha256.hexdigest()

In [ ]:
list(chemin_destination.iterdir())

In [ ]:
a = chemin_destination / "20240315_Tt_Bestof"
list(a.iterdir())

In [ ]:
photos_test = a.rglob("*.jpg")

resultats = {}
hashcodes = {}

for photo in photos_test:
    # Récupération des métadonnées EXIF
    img = Image.open(photo)
    exif_data = img._getexif()

    if exif_data:
        resultats[photo.name] = {
            TAGS.get(tag_id, tag_id): value for tag_id, value in exif_data.items()
        }

    # Calcul du hashcode
    hashcodes[photo.name] = calculer_hash(photo)

In [ ]:
resultats

In [ ]:
hashcodes

In [ ]:
import pandas as pd

df_resultats = pd.DataFrame.from_dict(resultats, orient="index")
df_hashcodes = pd.DataFrame.from_dict(hashcodes, orient="index")

print(df_resultats.shape)
print(df_hashcodes.shape)

In [ ]:
df_merge = pd.merge(
    df_resultats, df_hashcodes, left_index=True, right_index=True, how="left"
)

df_merge.head()

In [ ]:
df_merge = df_merge.rename(columns={0: "sha_256"})
df_merge.head()

In [ ]:
df = df_merge.copy()
df

In [ ]:
df["chemin"] = fichier_source.stem
df["name"] = df.index.astype(str)
df["cote"] = df["name"].str[1]
df["individu"] = df["name"].str[3:11]

df

In [ ]:
df.columns

In [ ]:
df_final = df[
    [
        "chemin",
        "individu",
        "cote",
        "DateTimeOriginal",
        "ExifImageWidth",
        "ExifImageHeight",
        "sha_256",
    ]
]

df_final = df_final.rename(
    columns={
        "DateTimeOriginal": "date",
        "ExifImageWidth": "largeur",
        "ExifImageHeight": "hauteur",
    }
)

df_final.head(2)